In [1]:
'''import ollama
from ollama import chat'''
from openai import OpenAI
import json

In [ ]:
class EvaluationAgent:
    def __init__(self, model_name: str="llama3"):
        self.model_name = model_name
        self.system_prompt = '''You are a Chief Examiner for Indian school examinations. Focus on technical keywords and curriculum specific accuracy.
        Follow a strict but fair philosophy to grading.
        Adhere to the following constraints:
        {
            "output_format": "JSON",
            "max_length": 5000
        }
        Follow this JSON format for responses:
        {
            "reasoning": "Justification and logic for the marks awarded, with explanation",
            "concepts_in_question": ["List the topics covered by the question"],
            "concepts_mastered": ["List the topics of the question mastered by the student as per the answer"],
            "knowledge_gap": ["List the topics which have been missed by the student in the answer"],
            "misconceptions": ["Identify the ideas that the student has misunderstood as per the answer"],
            "feedback_for_improvement": ["Targeted feedback to be used by the student to improve in future examinations"]
            "marks_awarded" : number,
            "total_possible_marks": number,
        }
        '''
    
    def evaluate(self, json_payload: str) -> str:
        try:
            client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key="sk-or-v1-"
            )
            data = json.loads(json_payload)
            question = data.get("Question", "")
            answer = data.get("Answer", "")
            subject = data.get("Subject", "General Subject")
            grade = data.get("Grade", "High School")
            board = data.get("Board", "Indian Educational Board")
            max_marks = data.get("Marks", 10)
            key = data.get("AnswerKey", "")
            user_content = f"""
            As a subject matter expert in {subject}, grade the student's answer for the {board} {grade}th grade exam.
            Use the provided answer key as the benchmark for grading the answer. 
            Your response must be formatted in JSON.
            Question: {question}
            Answer: {answer}
            Answer Key: {key}
            Subject: {subject}
            Board: {board}
            Grade: {grade}
            Maximum Marks: {max_marks}
            """
            response = client.chat.completions.create(model = self.model_name, messages=[
                {
                    'role' : 'system',
                    'content' : self.system_prompt
                },
                {
                    'role' : 'user',
                    'content' : user_content
                }
            ],
            response_format={"type": "json_object"},
            )
            return response.choices[0].message.content
        except json.JSONDecodeError:
            return json.dumps({"error" : "Failed to parse input"})
        except Exception as e:
            return json.dumps({"error" : str(e)})

In [ ]:
class ModerationAgent:
    def __init__(self, model_name: str="llama3"):
        self.model_name = model_name
        self.system_prompt = """
        You are an Expert Moderator of examiners for Indian school examinations. Compare the evaluation reports given by the two examiners and analyze the similarities and differences between the two.
        Analyse the reasoning and feedback provided by the examiners and point out any discrepancies.
        Adhere to the following constraints.
        {
            "output_format": "JSON",
            "max_length": 5000
        }
        Follow this JSON format for responses:
        {
            "feedback_evaluator_1": "Critical analysis and feedback on the report from evaluator 1",
            "feedback_evaluator_2": "Critical analysis and feedback on the report from evaluator 2",
            "reliable_evaluator_reasoning": ["List of reasons why one evaluator is chosen to be the final evaluator"],
            "chosen_evaluator": "Single number choosing the reliable evaluator"
        }
        """
    
    def moderate(self, json_payload: str) -> str:
        try:
            client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key="sk-or-v1-"
            )
            data = json.loads(json_payload)
            question = data.get("Question", "")
            answer = data.get("Answer", "")
            subject = data.get("Subject", "General Subject")
            grade = data.get("Grade", "High School")
            board = data.get("Board", "Indian Educational Board")
            max_marks = data.get("Marks", 10)
            key = data.get("AnswerKey", "")
            evaluator1 = data.get("Evaluator1", "")
            evaluator2 = data.get("Evaluator2", "")
            marks1 = data.get("Marks1", "")
            marks2 = data.get("Marks2", "")
            user_content = f"""
            As a subject matter expert in {subject} and an experienced moderator of evaluators, consider the following evaluations of the student answer for {board} {grade}th examination.
            Base your decisions and analysis on the feedback provided by the two evaluators, and the marks awarded by them.
            Your response must be formatted in JSON.
            Question: {question}
            Answer: {answer}
            Answer Key: {key}
            Subject: {subject}
            Board: {board}
            Grade: {grade}
            Maximum Marks: {max_marks}

            Evaluator 1:
            Marks awarded: {marks1}
            Justification: {evaluator1}

            Evaluator 2:
            Marks awarded: {marks2}
            Justification: {evaluator2}
            """
            response = client.chat.completions.create(model = self.model_name, messages=[
                {
                    'role' : 'system',
                    'content' : self.system_prompt
                },
                {
                    'role' : 'user',
                    'content' : user_content
                }
            ],
            response_format={"type": "json_object"},
            )
            return response.choices[0].message.content
        except json.JSONDecodeError:
            return json.dumps({"error" : "Failed to parse input"})
        except Exception as e:
            return json.dumps({"error" : str(e)})

In [4]:
agent1 = EvaluationAgent(model_name = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free")
agent2 = EvaluationAgent(model_name = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free")
moderator = ModerationAgent(model_name = "inclusionai/ring-2.6-1t:free")

In [ ]:
request = json.dumps({
    "Question" : """
    Two resistors, $R_1 = 6 \Omega$ and $R_2 = 3 \Omega$, are connected in parallel. This parallel combination is then connected in series with a third resistor, $R_3 = 8 \Omega$, and a 12V battery.
    Calculate:
    1- The equivalent resistance of the entire circuit.
    2- The total current flowing through the circuit.
    """,
    "Answer" : """
    Part 1: Equivalent Resistance
        First, calculate the equivalent resistance of the parallel combination (Rp) of R1 and R2:
        1/Rp = 1/R1 + 1/R2
        1/Rp = 1/6 + 1/3
        1/Rp = (1 + 2) / 6 = 3/6
        Rp = 2 ohms
        Next, calculate the total equivalent resistance (Req) by adding the series resistor (R3) to the parallel combination (Rp):
        Req = Rp + R3
        Req = 2 + 8 = 10 ohms
        The equivalent resistance of the entire circuit is 10 ohms.
        Part 2: Total Current
        Use Ohm's Law to find the total current (I) flowing through the circuit, given the total voltage (V = 12V) and the total equivalent resistance (Req = 10 ohms):
        V = I * Req
        I = V / Req
        I = 12 / 10 = 1.2A
        The total current flowing through the circuit is 1.2A.    
    """,
    "AnswerKey" : """
    1- The equivalent resistance of the entire circuit is $10 \Omega$. (Calculation: $R_p = \frac{6 \cdot 3}{6 + 3} = 2 \Omega$; $R_{eq} = 2 \Omega + 8 \Omega = 10 \Omega$)
    2- The total current flowing through the circuit is $1.2 \text{A}$. (Calculation: $I = \frac{12\text{V}}{10 \Omega} = 1.2 \text{A}$)
    """,
    "Subject" : "Physics",
    "Board": "CBSE",
    "Grade" : 10,
    "Marks" : 10
})
max = request.get("Marks")
response1 = agent1.evaluate(request)
response2 = agent2.evaluate(request)
scores = []
scores.append(json.loads(response1).get("marks_awarded"))
scores.append(json.loads(response2).get("marks_awarded"))
feedback = []
feedback.append(json.loads(response1).get("reasoning"))
feedback.append(json.loads(response2).get("reasoning"))
moderatorRequest = json.dumps({
    "Question" : """
    Two resistors, $R_1 = 6 \Omega$ and $R_2 = 3 \Omega$, are connected in parallel. This parallel combination is then connected in series with a third resistor, $R_3 = 8 \Omega$, and a 12V battery.
    Calculate:
    1- The equivalent resistance of the entire circuit.
    2- The total current flowing through the circuit.
    """,
    "Answer" : """
    Part 1: Equivalent Resistance
        First, calculate the equivalent resistance of the parallel combination (Rp) of R1 and R2:
        1/Rp = 1/R1 + 1/R2
        1/Rp = 1/6 + 1/3
        1/Rp = (1 + 2) / 6 = 3/6
        Rp = 2 ohms
        Next, calculate the total equivalent resistance (Req) by adding the series resistor (R3) to the parallel combination (Rp):
        Req = Rp + R3
        Req = 2 + 8 = 10 ohms
        The equivalent resistance of the entire circuit is 10 ohms.
        Part 2: Total Current
        Use Ohm's Law to find the total current (I) flowing through the circuit, given the total voltage (V = 12V) and the total equivalent resistance (Req = 10 ohms):
        V = I * Req
        I = V / Req
        I = 12 / 10 = 1.2A
        The total current flowing through the circuit is 1.2A.    
    """,
    "AnswerKey" : """
    1- The equivalent resistance of the entire circuit is $10 \Omega$. (Calculation: $R_p = \frac{6 \cdot 3}{6 + 3} = 2 \Omega$; $R_{eq} = 2 \Omega + 8 \Omega = 10 \Omega$)
    2- The total current flowing through the circuit is $1.2 \text{A}$. (Calculation: $I = \frac{12\text{V}}{10 \Omega} = 1.2 \text{A}$)
    """,
    "Subject" : "Physics",
    "Board": "CBSE",
    "Grade" : 10,
    "Marks" : 10,
    "Evaluator1" : f"{feedback[0]}",
    "Evaluator2" : f"{feedback[1]}",
    "Marks1" : f"{scores[0]}",
    "Marks2" : f"{scores[1]}"

})
if (abs(scores[0]/max - scores[1]/max) >= 0.25):
    moderatorOutput = moderator.moderate(moderatorRequest)
    print(moderatorOutput)
#output = json.loads(output['content'])
#print(json.dumps(output, indent=4))


{
  "feedback_evaluator_1": "Evaluator 1 correctly verified each step: parallel resistance formula, arithmetic (Rp = 2 Ω), series addition (Req = 10 Ω), and Ohm’s law for current (I = 1.2 A). The justification notes correct units and that all calculations are shown. However, it does not mention the handling of significant figures or the broader conceptual mastery, which could be relevant for a complete evaluation.",
  "feedback_evaluator_2": "Evaluator 2 also confirms the accurate application of the parallel‑resistor formula, the series addition, and Ohm’s law. The report additionally highlights the correct use of units and significant figures, and explicitly states that the student demonstrated full mastery of the required concepts. This provides a slightly more detailed assessment of the student’s understanding.",
  "reliable_evaluator_reasoning": [
    "Both evaluators agree on the correctness of the calculations and award the maximum marks.",
    "Evaluator 2 adds a specific commen

### Old Requests with old format

In [ ]:
req2 = json.dumps(
    {
    "Question": """
    The archaeological records provide no immediate answer of the existence of centre
    of power”. Give suitable examples to prove the statement in the context of Harappa. 
    """,
    "Answer": """
    There is no doubt that that the archaeological findings provide no immediate
    answer to the Harappa’s central authority. Many views have been given
    regarding the central authority. Following are some of them:
● A large building found at Mohenjodaro but no spectacular finds were
associated with it.
● A stone statue was found at the site of Mohenjodaro which have been
labelled as the ‘priest king’.
    """,
    "AnswerKey" : """
    There is no doubt that that the archaeological findings provide no immediate
answer to the Harappa’s central authority. Many views have been given
regarding the central authority. Following are some of them:
● Alarge building found at Mohenjodaro but no spectacular finds were
associated with it.
● A stone statue was found at the site of Mohenjodaro which have been
labelled as the ‘priest king’.
● But so far, the ritual practices of Harappan people have not been
understood.
● There is no clear evidence to know whether those who performed ritual
practices also held some political power.
Any three points to be described.
    """,
    "Subject": "History",
    "Board": "CBSE",
    "Grade": 12,
    "Marks": 3
}
)
response = agent.evaluate(req2)
output = json.loads(response)
#output = json.loads(output['content'])
print(json.dumps(output, indent=4))

In [ ]:
req3 = json.dumps(
    {
    "Question": """
    “Abdur Razzaq, an ambassador sent by the ruler of Persia to Calicut (present-day
    Kozhikode) in the fifteenth century, was greatly impressed by the fortifications, and
    mentioned seven lines of forts.” Substantiate the statement with suitable examples.
    """,
    "Answer": """ 
    The first dynasty, known as the Sangama dynasty, exercised control till 1485.
They were supplanted by the Saluvas, military commanders, who remained in
power till 1503 when they were replaced by the Tuluvas.
 Krishnadeva Raya belonged to the Tuluva dynasty. He ruled from 1509 till
1529 C.E. Following were his main achievements:
● The land between the Tungabhadra and Krishna rivers (the Raichur doab)
was acquired in1512.
● The rulers of Orissa were subdued in 1514 and severe defeats were
inflicted on the Sultan of Bijapur in 1520.
● Although the kingdom remained in a constant state of military preparedness,
it flourished under conditions of unparalleled peace and prosperity.
● Krishnadeva Raya is credited with building some fine temples and adding
impressive gopurams to many important south Indian temples. He also
founded a suburban township near Vijayanagara called Nagalapuram after
his mother.
● Krishnadeva Raya, the most famous ruler of Vijayanagara, composed a
work on statecraft in Telugu known as the Amuktamalyada. It was written in
the Telgu language.
● Although the armies of the Sultans were responsible for the destruction of
the city of Vijayanagara, relations between the Sultans and the rayas were
not always or inevitably hostile, in spite of religious differences. Krishnadeva
Raya, for example, supported some claimants to power in the Sultanates.
● Many foreign travellers like Barbosa, Paes and Fernao Nuniz wrote about
the good administration and prosperity of the Vijayanagar kingdom.
● In the end, it is clear that Krishna D
    """,
    "AnswerKey" : """
    Abdur Razzaq, an ambassador sent by the ruler of Persia to Calicut (presentday Kozhikode) in the fifteenth century, was greatly impressed by the
fortifications, and mentioned seven lines of forts. It is clear from the following
details:
● Different parts of the city of Vijayanagara were enclosed with the great fortress
walls.
 These encircled not only the city but also its agricultural hinterland and
forests. The outermost wall linked the hills surrounding the city.
 The massive masonry construction was slightly tapered. No mortar or
cementing agent was employed anywhere in the construction.
 The stone blocks were wedge shaped, which held them in place, and the
inner portion of the walls was of earth packed with rubble.
 Square or rectangular bastions projected outwards.
 What was most significant about this fortification is that it enclosed
agricultural tracts.
 Abdur Razzaq noted that between the first, second and the third walls there
are cultivated fields, gardens and houses.
 A second line of fortification went round the inner core of the urban complex,
and a third line surrounded the royal centre, within which each set of major
buildings was surrounded by its own high walls.
 The fort was entered through well-guarded gates, which linked the city to the
major roads. Gateways were distinctive architectural features that often
defined the structures to which they regulated access.
 Archaeologists have studied roads within the city and those leading out from
it. These have been identified by tracing paths through gateways, as well as
by finds of pavements.
 Roads generally wound around through the valleys, avoiding rocky terrain.
 Some of the most important roads extended from temple gateways, and were
lined by bazaars.
Any eight points to be described .
    """,
    "Subject": "History",
    "Board": "CBSE",
    "Grade": 12,
    "Marks": 8
}
)
response = agent.evaluate(req3)
print(response)

In [ ]:
req4 = json.dumps(
    {
"Question": """
What do you understand by circuit training? How will a coach plan circuit training
sessions with 6 stations to develop the fitness of his new trainees? Explain. 
""",
"Answer": """ 
1. Meaning of Circuit Training
Circuit training is a type of workout method where a person does several different exercises one after the other. It is designed to improve the overall fitness and health of the body. In this training, you complete a task at one station and then immediately move to the next station with very little rest in between, which helps in making the body stronger and more active.

2. Planning the Session for New Trainees
When a coach is planning a session for new trainees, they need to be careful because the trainees are beginners. The coach will plan the workout by looking at how fit the trainees currently are. The coach will make sure the exercises are not too hard so the trainees don't get too tired or injured, and will set a good amount of time for them to work out and a good amount of time for them to rest.

3. The 6-Station Plan
To develop the fitness of the trainees, the coach will set up 6 different stations in the field or gym. The plan will look like this:
* Station 1: Upper Body Exercise - The coach will start by giving the trainees an exercise that works out the top half of their body to make their arms stronger.
* Station 2: Stomach Exercise - Next, the trainees will do an exercise that focuses on the middle part of the body to build up their core strength.
* Station 3: Lower Body Exercise - At the third station, the trainees will do a movement that uses their legs. This will help them build power for running and jumping.
* Station 4: Stamina Exercise - For this station, the coach will choose an activity that makes the trainees breathe very heavily and gets their heart rate up to improve their stamina.
* Station 5: Movement Exercise - Here, the trainees will do an exercise where they have to move around a lot to improve their speed and how well they can control their bodies.
* Station 6: Relaxing Exercise - At the final station, the coach will give them an easy exercise to help them cool down and relax their muscles after completing the whole circuit.

Conclusion:
By using this 6-station plan, the coach will ensure that the new trainees get a complete workout that targets all the different parts of their body without pushing them too hard.
""",
"AnswerKey" : """
6 Sample Stations (Exercises):
● 1. Push-ups
● 2. Squats
● 3. Skipping
● 4. Sit-ups
● 5. Shuttle runs
● 6. Plank hold
(Or any other suitable)
Components to be Developed in New Trainees:
● Strength
● Endurance
● Flexibility
● Speed
● Agility
● Coordination
(Explanation of each point along with a circuit)
""",
"Subject": "Physical Education",
"Board": "CBSE",
"Grade":12,
"Marks": 5
}
)
response = agent.evaluate(req4)
print(response)

In [ ]:
req5 = json.dumps(
    {
    "Question":"""
    Iron-EDTA complex in food fortification
Food fortification is defined as the practice of
adding vitamins and minerals to commonly
consumed foods during processing to increase
their nutritional value. It is a proven, safe and
cost-effective strategy for improving diets and
for the prevention and control of micronutrient
deficiencies. A food product (such as rice, wheat
flour, edible oil) that is fortified through the
addition of fortificants is called a “vehicle”.
In African and south Asian countries 40 percent of the population suffers from
anaemia. Average human needs nearly 10mg of iron daily. Iron
fortification may be useful in fighting iron deficiencies in humans.
Reduced iron and several iron salts have been used in the past as iron
fortification, however, not all are suitable for this purpose, in terms of
iron absorption. Recent studies have shown that beverages containing
sugar fortified with either Ferrous sulphate or Fe(III)- EDTA complex
have high rate of absorption of iron.
Ferrous sulphate as well as Fe(III)- EDTA is suitable to enrich sugar,
but while iron from ferrous sulphate is precipitated and poorly absorbed
when fortified sugar is added to beverages such as tea, Fe(III)- EDTA
reacts slowly with tea and iron is not precipitated for at least 24 hr.
Fe(III)-EDTA as iron fortification, has demonstrated so far, more
advantages than that observed from other iron salts, including ferrous
sulphate. But, EDTA is a chelating agent and its use in food technology
to prevent oxidative damage of food has been restricted. Excessive
consumption of EDTA can cause abdominal cramps, nausea, low
blood pressure and damage to kidneys. According to National Institute
of Health, it is unsafe to consume more than 3 g of EDTA per day or
continuously for more than 5 to 7 days.
The amount of EDTA necessary for 10 mg of iron fortification, is about
60 mg. This is within the safe limits and is comparable to the usual
amount added to the diet.
Based on the information provided above, answer the following
questions:
I. Why is Fe(III)-EDTA complex stable as compared to Ferrous
sulphate?
II. You are a doctor, working in Somalia. Will you recommend iron
fortified food to your patients? Support your answer with
references from the passage.
III. What is the denticity of the ligand in the Fe(III) EDTA complex.
Name the atom(s) through which it can bind to the central
metal ion.
    """,
    "Answer" : """ 
    I. EDTA is a chelating agent, it forms ringed complex with the
central metal ion and makes the complex stable.
    II. Yes, 40 percent of the population in Africa suffers from anaemia. Most
of the patients in Somalia are likely to be anaemic. Iron fortified
food will have increased the nutritional value. In the same
amount of food product the patient will get higher amount of the
micronutrient than present in natural product.
This will help reduce cases of iron deficiency in Somalia.
However, patients will be advised to consume the food product
according to the recommended safe limits of the fortificant.
    III. Denticity = 6, 2 Nitrogen and 4 oxygen are electron donors
    """,
    "AnswerKey" : """ 
    I. EDTA is a chelating agent, it forms ringed complex with the
central metal ion and makes the complex stable
    II. Yes, 40 percent of the population in Africa suffers from anaemia. Most
of the patients in Somalia are likely to be anaemic. Iron fortified
food will have increased the nutritional value. In the same
amount of food product the patient will get higher amount of the
micronutrient than present in natural product.
This will help reduce cases of iron deficiency in Somalia.
However, patients will be advised to consume the food product
according to the recommended safe limits of the fortificant.
OR
No, though 40percent of the population suffers from anaemia, iron
fortified food will be recommended to patients whose reports suggest iron deficiency. Iron fortified food will have increased the
nutritional value. In the same amount of food product the patient
will get higher amount of the micronutrient than present in
natural product.
This fortificant can cause other ill effects to the non- anaemic
population as well as could lead to higher levels of iron in the
body than required. 
    III. (a)6
2 Nitrogen and 4 oxygen are electron donors
    """,
    "Subject": "Chemistry",
    "Board": "CBSE",
    "Grade":12,
    "Marks": 4
}
)
response = agent.evaluate(req5)
output = json.loads(response)
#output = json.loads(output['content'])
print(json.dumps(output, indent=4))

In [ ]:
req6 = json.dumps(
    {
    "Question": """ 
    A research student spoke with two people, M and N to learn about their work-related
differences. On the basis of the interview conducted with both of them, the student
concludes that while person M was working in an organized sector, person N was an
employee of a workplace that was functioning in an unorganised way. Analyse the
key differences between the two sectors that must have enabled the research
student to come to this conclusion.
    """,
    "Answer": """ 
    I think the working conditions for Person M might be something like this:

Work Timings: They probably have more regular work and the hours are likely fixed, so they don't have to worry about it changing every day.

Rules: I believe the place where they work is maybe registered with the government, so they probably have to follow some legal things or acts, though I can't remember the exact names of the laws.

Security: There is some kind of job security, I guess. They probably won't just get fired for no reason and might get extra money if they stay late, which is called overtime, I think.

Extra Stuff: They might get things like paid leave or some medical help from the company. It seems like they could also get a pension or some money after they stop working.

Environment: The employer is supposed to keep the place clean and maybe give them water and stuff because of the rules.

On the other hand, Person N seems to be in a different situation:

Job Type: Their jobs are probably low-paid and might be a bit irregular. They might not have work every single day of the month.

Government Role: I don't think the government really looks at these jobs much. It's mostly outside their control, so there aren't many protections for the workers there.

No Perks: They probably don't get things like holidays or extra pay for working more hours. If they get sick, they probably just don't get paid for that day.

Leaving the Job: I guess they can be told to leave at any time if the boss doesn't need them anymore, like during times when there is no work to do.

Self-Work: Some of these people might be doing small things on their own, like selling things on the street or fixing stuff, where there are no real rules to follow.
    """,
    "AnswerKey" : """ 
    Working conditions of person M would have the following features:
Regular Employment: Workers have assured, regular work with fixed terms of
employment.
Government Regulation: Enterprises are registered with the government and
follow legal rules and regulations (e.g., Factories Act, Minimum Wages Act).
Security of Employment: Workers enjoy job security with clear working hours and
benefits.
Overtime Compensation: If workers work beyond regular hours, they are paid
overtime.
Employee Benefits: Workers receive benefits like paid leave, holidays, provident
fund, gratuity, and medical benefits.
Safe Working Conditions: Employers are required to provide safe working
environments (e.g., clean drinking water).
Retirement Benefits: Workers are entitled to pensions after retirement.
Formal Processes: The sector follows formal processes and procedures for
employment.
Working conditions of person N would have the following features:
Irregular Employment: Jobs are low-paid and often irregular, with no guarantee of
continuous work.
Lack of Government Regulation: The sector operates largely outside government
control, with few or no legal protections.
No Employee Benefits: Workers do not receive benefits like paid leave, overtime
pay, or medical benefits.
Job Insecurity: Employment is not secure; workers can be dismissed without
notice or reason.
Seasonal Work: Employment is often dependent on seasons, and workers may be
laid off during off-peak periods.
Informal Jobs: Many workers are self-employed, doing small jobs like street
vending or repair work.
Dependence on Employer: Employment conditions are influenced by the
employer's whims and needs.
No Legal Protections: There is little enforcement of rules or regulations related
to working conditions or benefits.
    """,
    "Subject": "Social Studies",
    "Board": "CBSE",
    "Grade": 10,
    "Marks": 5
}
)
response = agent.evaluate(req6)
output = json.loads(response)
#output = json.loads(output['content'])
print(json.dumps(output, indent=4))